# Bronze Job Snapshot Writer

Captures immutable snapshots of each ingestion job execution.

**Purpose**: Audit trail for all Bronze ingestion jobs  
**Pattern**: Append-only job metadata  
**No transformations**: Raw job state only

In [0]:
# Job execution parameters
dbutils.widgets.text("job_id", "", "Job ID")
dbutils.widgets.text("job_name", "", "Job Name")
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("run_id", "", "Run ID")
dbutils.widgets.dropdown("status", "running", ["running", "success", "failed", "partial"], "Status")
dbutils.widgets.text("config_json", "{}", "Configuration JSON")
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

# Get parameter values
job_id = dbutils.widgets.get("job_id")
job_name = dbutils.widgets.get("job_name")
batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")
status = dbutils.widgets.get("status")
config_json = dbutils.widgets.get("config_json")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [0]:
%sql
-- Create job snapshot table if not exists
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.bronze_job_snapshot (
  snapshot_id STRING NOT NULL,
  job_id STRING NOT NULL,
  job_name STRING,
  batch_id STRING,
  run_id STRING,
  status STRING NOT NULL,
  config_json STRING,
  snapshot_timestamp TIMESTAMP NOT NULL,
  ingestion_timestamp TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Immutable snapshots of Bronze ingestion job executions'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true'
);

In [0]:
from pyspark.sql import functions as F
from uuid import uuid4

# Create snapshot record
snapshot_df = spark.createDataFrame([(
    str(uuid4()),
    job_id,
    job_name,
    batch_id,
    run_id,
    status,
    config_json,
    F.current_timestamp(),
    F.current_timestamp()
)], schema="snapshot_id STRING, job_id STRING, job_name STRING, batch_id STRING, run_id STRING, status STRING, config_json STRING, snapshot_timestamp TIMESTAMP, ingestion_timestamp TIMESTAMP")

# Append to Bronze table
target_table = f"{catalog}.{schema}.bronze_job_snapshot"
snapshot_df.write.mode("append").saveAsTable(target_table)

print(f"✓ Job snapshot written to {target_table}")
print(f"  Job: {job_name} ({job_id})")
print(f"  Batch: {batch_id}")
print(f"  Run: {run_id}")
print(f"  Status: {status}")